In [3]:
import os
import pandas as pd
import numpy as np 
from xgboost import XGBRegressor  

from sklearn.model_selection import KFold ,cross_val_score, cross_val_predict
from tqdm import tqdm 
from sklearn.model_selection import GridSearchCV , KFold , GridSearchCV

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler , MinMaxScaler

from sklearn.compose import make_column_selector, make_column_transformer
import warnings
warnings.filterwarnings("ignore")

os.chdir(r"C:/Users/RAJESH/Desktop/ss/Datasets/Cases/Chemical_Process_Data/")

In [4]:
chem = pd.read_csv('ChemicalProcess.csv')
chem

,Yield,BiologicalMaterial01,BiologicalMaterial02,BiologicalMaterial03,BiologicalMaterial04,BiologicalMaterial05,BiologicalMaterial06,BiologicalMaterial07,BiologicalMaterial08,BiologicalMaterial09,BiologicalMaterial10,BiologicalMaterial11,BiologicalMaterial12,ManufacturingProcess01,ManufacturingProcess02,ManufacturingProcess03,ManufacturingProcess04,ManufacturingProcess05,ManufacturingProcess06,ManufacturingProcess07,ManufacturingProcess08,ManufacturingProcess09,ManufacturingProcess10,ManufacturingProcess11,ManufacturingProcess12,ManufacturingProcess13,ManufacturingProcess14,ManufacturingProcess15,ManufacturingProcess16,ManufacturingProcess17,ManufacturingProcess18,ManufacturingProcess19,ManufacturingProcess20,ManufacturingProcess21,ManufacturingProcess22,ManufacturingProcess23,ManufacturingProcess24,ManufacturingProcess25,ManufacturingProcess26,ManufacturingProcess27,ManufacturingProcess28,ManufacturingProcess29,ManufacturingProcess30,ManufacturingProcess31,ManufacturingProcess32,ManufacturingProcess33,ManufacturingProcess34,ManufacturingProcess35,ManufacturingProcess36,ManufacturingProcess37,ManufacturingProcess38,ManufacturingProcess39,ManufacturingProcess40,ManufacturingProcess41,ManufacturingProcess42,ManufacturingProcess43,ManufacturingProcess44,ManufacturingProcess45
0,38.00,6.25,49.58,56.97,12.74,19.51,43.73,100.0,16.66,11.44,3.46,138.09,18.83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.00,NaN,NaN,NaN,35.5,4898.0,6108,4682,35.5,4865,6049,4665,0.0,NaN,NaN,NaN,4873.0,6074.0,4685.0,10.7,21.0,9.9,69.1,156,66.0,2.4,486.0,0.019,0.5,3,7.2,NaN,NaN,11.6,3.0,1.8,2.4
1,42.44,8.01,60.97,67.48,14.65,19.36,53.14,100.0,19.04,12.55,3.46,153.67,21.05,0.0,0.0,NaN,917.0,1032.2,210.0,177.0,178.0,46.57,NaN,NaN,0.0,34.0,4869.0,6095,4617,34.0,4867,6097,4621,0.0,3.0,0.0,3.0,4869.0,6107.0,4630.0,11.2,21.4,9.9,68.7,169,66.0,2.6,508.0,0.019,2.0,2,7.2,0.1,0.15,11.1,0.9,1.9,2.2
2,42.03,8.01,60.97,67.48,14.65,19.36,53.14,100.0,19.04,12.55,3.46,153.67,21.05,0.0,0.0,NaN,912.0,1003.6,207.1,178.0,178.0,45.07,NaN,NaN,0.0,34.8,4878.0,6087,4617,34.8,4877,6078,4621,0.0,4.0,1.0,4.0,4897.0,6116.0,4637.0,11.1,21.3,9.4,69.3,173,66.0,2.6,509.0,0.018,0.7,2,7.2,0.0,0.00,12.0,1.0,1.8,2.3
3,41.42,8.01,60.97,67.48,14.65,19.36,53.14,100.0,19.04,12.55,3.46,153.67,21.05,0.0,0.0,NaN,911.0,1014.6,213.3,177.0,177.0,44.92,NaN,NaN,0.0,34.8,4897.0,6102,4635,34.8,4872,6073,4611,0.0,5.0,2.0,5.0,4892.0,6111.0,4630.0,11.1,21.3,9.4,69.3,171,68.0,2.5,496.0,0.018,1.2,2,7.2,0.0,0.00,10.6,1.1,1.8,2.1
4,42.49,7.47,63.33,72.25,14.02,17.91,54.66,100.0,18.22,12.80,3.05,147.61,21.05,10.7,0.0,NaN,918.0,1027.5,205.7,178.0,178.0,44.96,NaN,NaN,0.0,34.6,4992.0,6233,4733,33.9,4886,6102,4659,-0.7,8.0,4.0,18.0,4930.0,6151.0,4684.0,11.3,21.6,9.0,69.4,171,70.0,2.5,468.0,0.017,0.2,2,7.3,0.0,0.00,11.0,1.1,1.7,2.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,39.66,6.71,56.32,66.19,12.35,20.02,50.26,100.0,17.54,12.50,2.82,143.45,20.32,12.8,21.5,1.54,935.0,1027.0,206.2,178.0,177.0,46.78,9.6,9.6,0.0,33.9,4829.0,6026,4621,33.5,4815,6011,4612,-0.4,2.0,2.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,156,NaN,NaN,NaN,NaN,2.3,0,0.0,0.0,0.00,0.0,0.6,0.0,0.0
172,39.68,6.87,56.74,66.61,12.55,20.18,50.80,100.0,17.48,12.41,2.82,143.10,20.24,12.8,21.5,1.56,933.0,1032.0,206.6,177.0,178.0,46.51,9.5,9.9,0.0,34.0,4833.0,6029,4608,33.5,4807,6001,4584,-0.5,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,158,NaN,NaN,NaN,NaN,1.0,0,0.0,0.0,0.00,0.0,0.6,0.0,0.0
173,42.23,7.50,58.41,68.30,13.33,20.81,52.96,100.0,17.23,12.04,2.83,141.72,19.92,13.0,20.4,1.55,930.0,1040.0,208.7,178.0,177.0,48.05,10.1,10.4,0.0,33.1,4795.0,6000,4557,32.8,4764,5977,4538,-0.3,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,167,NaN,NaN,NaN,NaN,1.3,0,0.0,0.0,0.00,0.0,0.6,0.0,0.0
174,38.48,7.53,58.36,69.25,14.35,20.57,51.31,100.0,17.87,12.77,3.55,145.56,20.04,14.1,21.6,1.55,935.0,1044.8,208.0,177.0,

In [6]:
X , y = chem.drop('Yield',axis = 1) , chem['Yield']

In [13]:


scl_scaler , mm_scaler = StandardScaler(), MinMaxScaler()

imputer  = SimpleImputer()
knn = KNeighborsRegressor(n_neighbors=1)
kfold = KFold(random_state=26, n_splits=5, shuffle=True)

pipe = Pipeline([("IMP" , imputer),("SCL", scl_scaler),('KNN', knn)])
# pipe.get_params()

In [9]:
xgb = XGBRegressor(random_state=26)


params = {
    "IMP__strategy":['mean','median'],
    "SCL":[scl_scaler, mm_scaler, None],
    "KNN__n_neighbors": np.arange(1,11)
}
params

{'IMP__strategy': ['mean', 'median'],
 'SCL': [StandardScaler(), MinMaxScaler(), None],
 'KNN__n_neighbors': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])}

In [11]:
gcv = GridSearchCV(pipe,param_grid = params , cv = kfold, scoring='r2',n_jobs=-1)
gcv.fit(X,y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...eighbors=1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'IMP__strategy': ['mean', 'median'], 'KNN__n_neighbors': array([ 1, 2..., 8, 9, 10]), 'SCL': [StandardScaler(), MinMaxScaler(), ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Su

In [12]:
print(f'Best Params : {gcv.best_params_}')
print(f'Best Score : {gcv.best_score_}')

Best Params : {'IMP__strategy': 'mean', 'KNN__n_neighbors': np.int64(2), 'SCL': StandardScaler()}
Best Score : 0.5092484572934366
